# Agent的高级用法-流式输出

我们可以根据不同的目标来选择不同的输出模式。例如：

实现 实时对话交互 ，优先选择messages模式；

观察Agent的 思考与执行步骤 ，优先选择updates模式；

需要查看 每一步状态 优先选择values/tasks/debug模式；

在工具执行时 输出自定义业务 日志优先选择custom模式


## 1、values输出模式

当`stream_mode`设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。

In [1]:
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [4]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages" : [
            # {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
             HumanMessage("查询客户id为cust1234的完整的信息、历史订单和可用优惠")
        ]
    },
    stream_mode="values"
):
    rprint(chunk)
    print("-" * 50)


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='82ab695f-33e4-4e26-ab7d-fc528fd01b6e'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='82ab695f-33e4-4e26-ab7d-fc528fd01b6e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 69,
                    'prompt_tokens': 276,
                    'total_tokens': 345,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 38,
                        'engine_ttlt_ms': 288,
                        'pre_inference_ms': 96,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 398,
                        'service_ttlt_ms': 602,
                        'user_visible_ttft_ms': 302
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-EgR9mLiqLhLKuocMeMLHdemHTKq0x',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1a5ca-7e69-76a5-97b3-719d57924471-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_m5VQrKI1Ubtzjg56GyZl5XDN',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_sVfPqAatc1BwYpFuXkVaQ6P4',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_ecX6LV93wlXIEo4iAoC30cB9',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 276,
                'output_tokens': 69,
                'total_tokens': 345,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='82ab695f-33e4-4e26-ab7d-fc528fd01b6e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 69,
                    'prompt_tokens': 276,
                    'total_tokens': 345,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 38,
                        'engine_ttlt_ms': 288,
                        'pre_inference_ms': 96,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 398,
                        'service_ttlt_ms': 602,
                        'user_visible_ttft_ms': 302
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-EgR9mLiqLhLKuocMeMLHdemHTKq0x',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1a5ca-7e69-76a5-97b3-719d57924471-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_m5VQrKI1Ubtzjg56GyZl5XDN',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_sVfPqAatc1BwYpFuXkVaQ6P4',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_ecX6LV93wlXIEo4iAoC30cB9',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 276,
                'output_tokens': 69,
                'total_tokens': 345,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='9c5a99ed-b0f8-488b-a597-1f5c339f06ca',
            tool_call_id='call_m5VQrKI1Ubtzjg56GyZl5XDN'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='15c0146d-edd7-4217-a7c5-220d64c84351',
            tool_call_id='call_sVfPqAatc1BwYpFuXkVaQ6P4'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='7bcfb858-53ce-4f93-b512-b22cec2e97b9',
            tool_call_id='call_ecX6LV93wlXIEo4iAoC30cB9'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='82ab695f-33e4-4e26-ab7d-fc528fd01b6e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 69,
                    'prompt_tokens': 276,
                    'total_tokens': 345,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 4,
                        'engine_ttft_ms': 38,
                        'engine_ttlt_ms': 288,
                        'pre_inference_ms': 96,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 398,
                        'service_ttlt_ms': 602,
                        'user_visible_ttft_ms': 302
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-EgR9mLiqLhLKuocMeMLHdemHTKq0x',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1a5ca-7e69-76a5-97b3-719d57924471-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_m5VQrKI1Ubtzjg56GyZl5XDN',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_sVfPqAatc1BwYpFuXkVaQ6P4',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_ecX6LV93wlXIEo4iAoC30cB9',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 276,
                'output_tokens': 69,
                'total_tokens': 345,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='9c5a99ed-b0f8-488b-a597-1f5c339f06ca',
            tool_call_id='call_m5VQrKI1Ubtzjg56GyZl5XDN'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='15c0146d-edd7-4217-a7c5-220d64c84351',
            tool_call_id='call_sVfPqAatc1BwYpFuXkVaQ6P4'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='7bcfb858-53ce-4f93-b512-b22cec2e97b9',
            tool_call_id='call_ecX6LV93wlXIEo4iAoC30cB9'
        ),
        AIMessage(
            content='客户 **cust1234** 的信息如下：\n\n### 1) 客户基本信息\n- 姓名：张三\n- 等级：VIP\n- 
加入日期：2023-01-15\n\n### 2) 历史订单\n- 总订单数：15\n- 总花费：25800.0\n\n### 3) 当前可用优惠\n- 老用户优惠\n- 
会员专属折扣\n- 有效期至：2027-01-31\n\n如果你需要，我还可以帮你把这些信息整理成一份更适合导出的客户档案摘要。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage'

--------------------------------------------------


## 2、updates输出模式

这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [5]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

{
    'model': {
        'messages': [
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 69,
                        'prompt_tokens': 276,
                        'total_tokens': 345,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                        'latency_checkpoint': {
                            'engine_tbt_ms': 4,
                            'engine_ttft_ms': 53,
                            'engine_ttlt_ms': 307,
                            'pre_inference_ms': 98,
                            'service_tbt_ms': 4,
                            'service_ttft_ms': 336,
                            'service_ttlt_ms': 579,
                            'user_visible_ttft_ms': 238
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5.4-mini-2026-03-17',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-EgRCGjcKyfyeYMB2RRcfn9HihHl8b',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1a5cc-b264-7399-b73b-b9247daaa6b0-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_VMiwuObprX8O4Pxew11nEsfL',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_CCVyniqsQR05lAqLfybIFmHo',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_jrqlghdZrYY01Ir2zI6tLwXO',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 276,
                    'output_tokens': 69,
                    'total_tokens': 345,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 0}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='65d2f147-f2fe-4549-9e42-77f7bedfa937',
                tool_call_id='call_jrqlghdZrYY01Ir2zI6tLwXO'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='2cb3d68f-4814-4817-8fc4-db0234133bfe',
                tool_call_id='call_VMiwuObprX8O4Pxew11nEsfL'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='48498c91-b83d-4293-b55e-d823bf31f0e3',
                tool_call_id='call_CCVyniqsQR05lAqLfybIFmHo'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='客户 **cust1234** 的信息如下：\n\n### 1) 客户基本信息\n- 姓名：张三\n- 等级：VIP\n- 
加入日期：2023-01-15\n\n### 2) 历史订单\n- 总订单数：15\n- 总花费：25800.0\n\n### 3) 当前可用优惠\n- 老用户优惠\n- 
会员专属折扣\n- 有效期至：2027-01-31\n\n如果你需要，我还可以帮你把这些信息整理成一份更适合汇报/导出的格式。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 134,
                        'prompt_tokens': 431,
                        'total_tokens': 565,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                        'latency_checkpoint': {
                            'engine_tbt_ms': 3,
                            'engine_ttft_ms': 29,
                            'engine_ttlt_ms': 486,
                            'pre_inference_ms': 101,
                            'service_tbt_ms': 3,
                            'service_ttft_ms': 367,
                            'service_ttlt_ms': 789,
                            'user_visible_ttft_ms': 266
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5.4-mini-2026-03-17',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-EgRCIzDELdy8KJywuwYMGN7aZcaPt',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a1a5cc-e02a-739d-bc7d-8d797d29a192-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 431,
                    'output_tokens': 134,
                    'total_tokens': 565,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 0}
                }
            )
        ]
    }
}

--------------------------------------------------


## 3、messages输出模式

该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似 ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [7]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="messages"
):
    # rprint(chunk)
    # print("-" * 50)
    print(chunk[0].content,end="",flush=True)

{"total_orders": 15, "total_spent": 25800.0}{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}客户 **cust1234** 的信息如下：

### 1) 客户基本信息
- 姓名：张三
- 等级：VIP
- 加入日期：2023-01-15

### 2) 历史订单
- 总订单数：15
- 总花费：25800.0

### 3) 当前可用优惠
- 老用户优惠
- 会员专属折扣
- 有效期至：2027-01-31

如果你需要，我还可以帮你把这些信息整理成更适合业务系统查看的表格格式。

## 4、tasks输出模式

该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。

In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="tasks"
):
    rprint(chunk)
    print("-" * 50)

## 5、debug输出模式

该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="debug"
):
    rprint(chunk)
    print("-" * 50)

## 6、checkpoints输出模式

该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# 其他工具代码同上，保持不变
# ... ...
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions],
    checkpointer=checkpointer  # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        config=config,
        stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    print(chunk)
    print("-" * 50)

## 7、custom输出模式

开发者通过`get_stream_writer`在工具或节点内部`自定义发送的数据`，用于`输出`业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [9]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()

    writer({"type": "生成销售报告", "message": "开始生成销售报告"})

    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i * 25}%"})

    writer({"type": "生成销售报告", "message": "报告生成完成"})

    return f"销售报告：总收入150万元，同比增长12%"


@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")

    return "当前库存量为10000件，库存充足，无异常"


# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
        {"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},
        stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：25%'}
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：50%'}
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------
